# Exploratory Data Analysis: Saleh Nonlinear Power Amplifier & 16-QAM
**EC-417 Introduction to Machine Intelligence | Project P15**
Roll Numbers: 623160, 623192

This notebook visualizes:
1. Undistorted 16-QAM constellation grid.
2. Saleh AM/AM (amplitude compression) and AM/PM (phase rotation) curves.
3. Resulting distorted constellation received at the detector with AWGN.

In [ ]:
import os
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
from scripts.get_data import get_16qam_constellation, apply_saleh_pa, generate_samples

### 1. Saleh Model Characteristic Curves

In [ ]:
r = np.linspace(0, 1.5, 200)
alpha_a, beta_a = 2.0, 1.0
alpha_phi, beta_phi = np.pi / 3.0, 1.0

am_am = (alpha_a * r) / (1.0 + beta_a * (r ** 2))
am_pm = (alpha_phi * (r ** 2)) / (1.0 + beta_phi * (r ** 2)) * (180.0 / np.pi)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(r, am_am, 'b-', lw=2, label='Saleh AM/AM')
ax1.plot(r, 2.0 * r, 'k--', alpha=0.5, label='Linear (Slope=2.0)')
ax1.set_title('AM/AM Amplitude Compression')
ax1.set_xlabel('Input Magnitude |r|')
ax1.set_ylabel('Output Magnitude A(r)')
ax1.grid(True, linestyle=':')
ax1.legend()

ax2.plot(r, am_pm, 'r-', lw=2, label='Saleh AM/PM')
ax2.set_title('AM/PM Phase Rotation')
ax2.set_xlabel('Input Magnitude |r|')
ax2.set_ylabel('Phase Shift (degrees)')
ax2.grid(True, linestyle=':')
ax2.legend()

plt.tight_layout()
plt.show()

### 2. Constellation Warping: Ideal vs Saleh PA Output

In [ ]:
ideal_points = get_16qam_constellation()
distorted_centroids = apply_saleh_pa(0.7 * ideal_points)

# Generate received points with AWGN at 18 dB
X_noisy, y_noisy = generate_samples(2000, snr_db=18.0, seed=42)

plt.figure(figsize=(7, 7))
plt.scatter(X_noisy[:, 0], X_noisy[:, 1], c=y_noisy, cmap='tab20', alpha=0.4, s=15, label='Received (18 dB AWGN)')
plt.scatter(distorted_centroids.real, distorted_centroids.imag, c='black', marker='x', s=80, lw=2, label='Distorted Centroids')
plt.scatter(ideal_points.real * 0.88, ideal_points.imag * 0.88, c='red', marker='+', s=80, lw=2, label='Ideal Grid (Power Scaled)')

plt.title('16-QAM Constellation under Saleh PA Distortion')
plt.xlabel('In-Phase (I)')
plt.ylabel('Quadrature (Q)')
plt.legend(loc='upper right')
plt.grid(True, linestyle=':')
plt.axis('equal')
plt.show()